In [ ]:
import pandas as pd
import numpy as np
from gbmcollaboration import AllDataCenters, CHUVDataCenter, UKERDataCenter, CRDataCenter
from gbmcollaboration import (
    SURVIVAL_EXCLUSION
)

def load_wes():
    # -- Loop over the three data centers and merge the clinical data
    merged_df = []

    for center in [CHUVDataCenter, UKERDataCenter, CRDataCenter]:
        center_data_dict = center.load_tabular(
            exclude_idh1_mut=True,
            exclude_patient_ids=True,
            exclude_relapse=True
        )
        clin_dict = center_data_dict['clinical']
        df = clin_dict['processed gbm clinical']

        # To be sure check if we only have Baseline samples
        print(f"---Loading the clinical data from {center.name}---")
        print(df['Sample collection chronology'].value_counts())
        print(f"Found {len(df)} primary cases")
        # append to the merged dataframe
        merged_df.append(df)

    # merge the three dataframes
    df_concat = pd.concat(merged_df, ignore_index=False)
    print("Dataframes from the three centers have been merged.")

    return df_concat

In [ ]:
data_sp = AllDataCenters.load_spatial(
    exclude_low_quality_visium = True,
    exclude_idh1_mut = True,
    exclude_patient_ids = True,
    exclude_relapse = True,
    load_deconvolution = True,
    load_gex = False
)
sp_samples = np.hstack([v.celltype_df.index.get_level_values('sample').unique() for k,v in data_sp.items()])
sp_samples = pd.Index(sp_samples).str.replace('_vis','')
%xdel data_sp

In [ ]:
adata_sc = AllDataCenters.load_singlecell(
    exclude_idh1_mut=True,
    exclude_patient_ids = True,
    exclude_relapse = True,
)
sc_samples = pd.Index(adata_sc.obs['orig.ident'].unique()).astype(str)
# %xdel adata_sc

In [ ]:
df_wes = load_wes()
# Exclude CR samples for which survival data is pending
df_wes_surv = df_wes.drop(index=SURVIVAL_EXCLUSION)

wes_samples = df_wes.index.unique()
wes_surv_samples = df_wes_surv.index.unique()

In [ ]:
mod_samples = {
    "Visium":sp_samples,
    "snRNAseq":sc_samples,
    "WES":wes_samples,
    "WES_surv":wes_surv_samples
}

mod_patients = {
    k:v.str[:-1].unique() for k,v in mod_samples.items()
}

cohort_mod = {
    "samples":mod_samples,
    "patients":mod_patients
}
# Build dataframe with presence/absence
mod_dfs = {}
for lvl, mod_dict in cohort_mod.items():
    union = sorted(set().union(*mod_dict.values()))
    df = pd.DataFrame(index=union)
    for name, lst in mod_dict.items():
        df[name] = df.index.isin(lst)

    mod_dfs[lvl] = df

In [ ]:
mod_counts = mod_dfs['samples'].sum()
joint_mod_subsets = [
    ['Visium','snRNAseq'],
    ['Visium','WES'],
    ['Visium','WES_surv'],
    ['snRNAseq','WES'],
    ['snRNAseq','WES_surv'],
    ['Visium','snRNAseq','WES'],
    ['Visium','snRNAseq','WES_surv']]
for lst in joint_mod_subsets:
    mod_counts['_'.join(lst)] = mod_dfs['samples'][lst].all(1).sum()

In [ ]:
mod_counts